In [1]:
## Import Required Libraries
import pandas as pd
import numpy as np 
import pickle

## Content-Based Movie Recommendation using Bag of Words

This notebook implements a content-based recommendation system using the Bag of Words approach. The preprocessed movie tags are transformed into numerical vectors, and cosine similarity is used to identify movies with similar content.

In [2]:
# Load the preprocessed movie dataset.
new_df = pickle.load(open("new_df.pkl", "rb"))

In [3]:
# Preview the processed dataset.
new_df.head() 

,title,movie_id,tags
0,Avatar,19995,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,285,"captain barbossa, long believed to be dead, ha..."
2,Spectre,206647,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,49026,following the death of district attorney harve...
4,John Carter,49529,"john carter is a war-weary, former military ca..."


In [4]:
new_df["tags"][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

### Apply Stemming

Stemming reduces words to their root form by removing common suffixes. This helps group similar words (such as *connect*, *connected*, and *connecting*) under a common representation, reducing vocabulary size before vectorization.

In [5]:
# Import the Porter Stemmer.
from nltk.stem.porter import PorterStemmer 
ps = PorterStemmer() 

In [6]:
def stem(text): 
    L = [] 
    for el in text.split(): 
        L.append(ps.stem(el)) 
    return " ".join(L) 

In [7]:
# Apply stemming to the movie tags.
new_df["tags"] = new_df["tags"].apply(stem) 

### Vectorization using Bag of Words

The movie tags are converted into numerical feature vectors using the Bag of Words technique. To keep the feature space computationally efficient, only the 5000 most frequent terms are retained while common English stop words are removed.

In [8]:
# Import the CountVectorizer.
from sklearn.feature_extraction.text import CountVectorizer 

# Initialize the vectorizer with the 5000 most frequent terms.
cv = CountVectorizer(max_features=5000, stop_words="english") 

In [9]:
# Convert the movie tags into Bag of Words vectors.
vectors = cv.fit_transform(new_df["tags"]).toarray() 

In [10]:
# Display the shape of the generated feature matrix.
vectors.shape

(4806, 5000)

### Compute Cosine Similarity

Cosine similarity is used to measure the similarity between movie vectors. Movies with higher similarity scores are considered more relevant and are therefore recommended together.

In [11]:
# Import the cosine similarity function.
from sklearn.metrics.pairwise import cosine_similarity 

# Compute the cosine similarity matrix.
similarity = cosine_similarity(vectors) 

In [12]:
# Display the shape of the similarity matrix.
similarity.shape

(4806, 4806)

### Recommendation Function

The recommendation function retrieves the five most similar movies for a given movie title based on cosine similarity scores.

In [13]:
# Generate the top five movie recommendations.
def recommend(movie): 
    movie_index = new_df[new_df["title"] == movie].index[0] 

    similar_items = sorted(list(enumerate(similarity[movie_index])), key = lambda x: x[1], reverse=True)[1:6] 

    recommendations = [] 
    
    for el in similar_items: 
        recommendations.append((new_df.iloc[el[0]].title, new_df.iloc[el[0]].movie_id)) 
    return recommendations 

In [14]:
# Generate recommendations for a sample movie.
recommend("Avatar") 

[('Aliens vs Predator: Requiem', np.int64(440)),
 ('Aliens', np.int64(679)),
 ('Falcon Rising', np.int64(270938)),
 ('Independence Day', np.int64(602)),
 ('Titan A.E.', np.int64(7450))]

In [15]:
# Save the similarity matrix.
pickle.dump(similarity, open("similarity_BoW.pkl", "wb")) 